# Arrhythmia Types - ECG Graphs

ECG waveform graphs for the arrhythmia types used in the thesis.
Waveforms are generated with simple math (Gaussian P/QRS/T building
blocks) - for illustration only. Model training and testing use the real
recordings from the datasets in data/raw/.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 250  # sampling rate for the plots

def g(t, c, w):
    return np.exp(-0.5 * ((t - c) / w) ** 2)

def qrs(t, r, amp=1.0, wide=False):
    w = 0.055 if wide else 0.028
    return (-0.12 * g(t, r - 0.018, w) * amp + g(t, r, w * 0.8) * amp
            - 0.28 * g(t, r + 0.024, w) * amp)

def p_w(t, c):
    return 0.14 * g(t, c, 0.028)

def t_w(t, r, amp=0.22):
    return amp * g(t, r + 0.24, 0.07)

def show(t, sig, title):
    plt.figure(figsize=(12, 2.5))
    plt.plot(t, sig, lw=0.8, color='tab:blue')
    plt.xlabel('time (s)'); plt.ylabel('mV'); plt.grid(alpha=0.3)
    plt.title(title)
    plt.show()

def T(dur=10):
    return np.arange(0, dur, 1 / fs)

print('ready')

## 1. Normal Sinus Rhythm
Regular P-QRS-T, rate 60-100 bpm.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 0.8
show(t, sig, '1. Normal Sinus Rhythm (75 bpm)')

## 2. Sinus Bradycardia
Same as normal but slower than 60 bpm.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 1.2
show(t, sig, '2. Sinus Bradycardia (50 bpm)')

## 3. Sinus Tachycardia
Same as normal but faster than 100 bpm.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 0.5
show(t, sig, '3. Sinus Tachycardia (120 bpm)')

## 4. Atrial Fibrillation (AFib)
No P waves, irregular R-R intervals, small fibrillatory baseline.

In [ ]:
rng = np.random.default_rng(7)
t = T()
sig = 0.05 * rng.standard_normal(len(t)) + 0.05 * np.sin(2*np.pi*7*t)
r = 0.3
while r < 10:
    sig += qrs(t, r) + t_w(t, r, 0.18)
    r += rng.uniform(0.45, 1.15)   # irregular R-R
show(t, sig, '4. Atrial Fibrillation (irregular R-R, no P)')

## 5. Atrial Flutter
Sawtooth flutter (F) waves with a regular ventricular response.

In [ ]:
t = T()
saw = 2 * (t * 5 % 1) - 1
sig = 0.09 * saw
r = 0.3
while r < 10:
    sig += qrs(t, r) + t_w(t, r, 0.15)
    r += 0.4   # 150 bpm regular
show(t, sig, '5. Atrial Flutter (sawtooth F waves)')

## 6. Supraventricular Tachycardia (SVT)
Very fast, regular, narrow QRS. P waves not visible.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += qrs(t, r) + t_w(t, r, 0.15)
    r += 0.32   # ~187 bpm
show(t, sig, '6. SVT (fast regular narrow QRS)')

## 7. Ventricular Tachycardia (VT)
Run of fast, wide QRS complexes.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += qrs(t, r, amp=0.9, wide=True) + t_w(t, r, 0.25)
    r += 0.3   # 200 bpm
show(t, sig, '7. Ventricular Tachycardia (wide, fast)')

## 8. Ventricular Fibrillation (VFib)
Chaotic electrical activity, no organized QRS at all.

In [ ]:
rng = np.random.default_rng(3)
t = T()
wander = np.convolve(rng.standard_normal(len(t)), np.ones(20)/20, 'same')
sig = 0.5 * wander * (1 + 0.5*np.sin(2*np.pi*1.5*t))
show(t, sig, '8. Ventricular Fibrillation (chaotic)')

## 9. Accelerated Idioventricular Rhythm (AIVR)
Like VT but slower (40-100 bpm), wide QRS.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += qrs(t, r, amp=0.85, wide=True) + t_w(t, r, 0.25)
    r += 0.85   # ~70 bpm
show(t, sig, '9. AIVR (wide but slow)')

## 10. Torsades de Pointes (TdP)
VT where the QRS amplitude twists around the baseline.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    amp = np.sin(2*np.pi*0.3*(r))   # twisting amplitude
    sig += qrs(t, r, amp=amp, wide=True)
    r += 0.3
show(t, sig, '10. Torsades de Pointes (twisting points)')

## 11. PVC - Premature Ventricular Contraction
Early wide beat without a P wave, then a compensatory pause.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 0.8
# insert an early wide beat at t=4.1 with a pause after
sig += qrs(t, 4.1, amp=1.3, wide=True) + t_w(t, 4.1, 0.3)
show(t, sig, '11. PVC (early wide beat + pause, arrow area ~4s)')

## 12. PAC - Premature Atrial Contraction
Early beat that still has a P wave and a narrow QRS.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 0.8
sig += p_w(t, 4.26-0.16) + qrs(t, 4.26, 0.9) + t_w(t, 4.26, 0.2)
show(t, sig, '12. PAC (early narrow beat with P, ~4s)')

## 13. 1st-Degree AV Block
Every P conducts, but the PR interval is long (>200 ms).

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.5
while r < 10:
    sig += p_w(t, r-0.30) + qrs(t, r) + t_w(t, r)
    r += 0.9
show(t, sig, '13. 1st-Degree AV Block (long PR)')

## 14. 2nd-Degree AV Block (Wenckebach)
PR gets longer each beat until one QRS is dropped.

In [ ]:
t = T()
sig = np.zeros_like(t)
p = 0.4
pr_cycle = [0.16, 0.24, 0.32]
i = 0
r_next = None
while p < 10:
    sig += p_w(t, p)
    pr = pr_cycle[i % 3]
    if pr_cycle[i % 3] < 0.32:
        sig += qrs(t, p + pr) + t_w(t, p + pr)
    # 4th P has no QRS (dropped beat)
    p += 0.7
    i += 1
show(t, sig, '14. 2nd-Degree AV Block (PR lengthens, then dropped)')

## 15. 3rd-Degree (Complete) AV Block
P waves and QRS march independently; ventricular rate is slow.

In [ ]:
t = T()
sig = np.zeros_like(t)
p = 0.2
while p < 10:              # atrial rate ~85
    sig += p_w(t, p)
    p += 0.7
r = 0.5
while r < 10:              # ventricular rate ~40, wide
    sig += qrs(t, r, amp=0.9, wide=True) + t_w(t, r, 0.28)
    r += 1.5
show(t, sig, '15. Complete Heart Block (P and QRS dissociated)')

## 16. Bundle Branch Block (BBB)
Wide, notched QRS at a normal rate.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    q = qrs(t, r, amp=0.9, wide=True)
    q += 0.35 * g(t, r + 0.09, 0.05)   # late notched peak
    sig += p_w(t, r-0.16) + q + t_w(t, r, 0.28)
    r += 0.8
show(t, sig, '16. Bundle Branch Block (wide notched QRS)')

## 17. WPW Syndrome
Short PR with a slurred QRS upstroke (delta wave).

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:
    q = qrs(t, r)
    q += 0.30 * g(t, r - 0.035, 0.045)   # delta wave slurring
    sig += p_w(t, r-0.09) + q + t_w(t, r)
    r += 0.8
show(t, sig, '17. WPW (short PR + delta wave)')

## 18. Parasystole
A second ventricular focus fires on its own schedule,
overlapping the normal rhythm.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 10:              # normal sinus rhythm
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 0.8
f = 0.9
while f < 10:              # independent focus every 1.9 s
    sig += qrs(t, f, amp=1.1, wide=True) + t_w(t, f, 0.25)
    f += 1.9
show(t, sig, '18. Parasystole (two independent rhythms)')

## 19. Sick Sinus Syndrome (tachy-brady)
Alternating fast runs and long pauses / slow rhythm.

In [ ]:
t = T()
sig = np.zeros_like(t)
r = 0.3
while r < 4.2:             # fast phase
    sig += qrs(t, r) + t_w(t, r, 0.15)
    r += 0.4
r = 7.2                    # long pause, then slow phase
while r < 10:
    sig += p_w(t, r-0.16) + qrs(t, r) + t_w(t, r)
    r += 1.4
show(t, sig, '19. SSS tachy-brady (fast, pause, slow)')

## Note

All waveforms above are generated for illustration. For model training
and evaluation the project uses real recordings (MIT-BIH, PTB-XL,
Chapman-Shaoxing, afdb) - see docs/DATASETS.md.
